In [2]:

from pyspark.sql import SparkSession


In [3]:
spark = SparkSession.builder \
    .appName("Lab02") \
    .getOrCreate()

## Create Data

In [4]:

employees = spark.createDataFrame([
    (1, "Ali", 3000),
    (2, "Sara", 7000),
    (3, "Omar", None),
    (4, "Mona", 10000)
], ["id", "name", "salary"])

departments = spark.createDataFrame([
    (1, "HR"),
    (2, "IT"),
    (2, "IT_DUPLICATE"),
    (3, "Finance"),
], ["id", "department"])


employees.show()
departments.show()

+---+----+------+
| id|name|salary|
+---+----+------+
|  1| Ali|  3000|
|  2|Sara|  7000|
|  3|Omar|  null|
|  4|Mona| 10000|
+---+----+------+

+---+------------+
| id|  department|
+---+------------+
|  1|          HR|
|  2|          IT|
|  2|IT_DUPLICATE|
|  3|     Finance|
+---+------------+




## Join the two dataframe


In [ ]:

employees.join(departments, on="id", how="left")

id,name,salary,department
1,Ali,3000,HR
3,Omar,null,Finance
2,Sara,7000,IT_DUPLICATE
2,Sara,7000,IT
4,Mona,10000,null



### Debug Task:
- Why does id=2 appear multiple times?
- What kind of data issue caused this?


## Fix Join Issue

In [37]:
departments_cleaned = departments.dropDuplicates(["id"])
departments_cleaned.withColumnRenamed("IT_DUPLICATE","id")

df_joined = employees.join(departments_cleaned, on="id", how="inner")

df_joined.show()


+---+----+------+------------+
| id|name|salary|  department|
+---+----+------+------------+
|  1| Ali|  3000|          HR|
|  3|Omar|  null|     Finance|
|  2|Sara|  7000|IT_DUPLICATE|
+---+----+------+------------+



## categorize the employee salary with new column using Case When

In [41]:
employees.createOrReplaceTempView("emp_table")

employees_categorized = spark.sql("""
    SELECT *, 
           CASE 
               WHEN salary IS NULL THEN 'Unknown'
               WHEN salary < 5000 THEN 'Low'
               WHEN salary <= 8000 THEN 'Medium'
               ELSE 'High'
           END as salary_category
    FROM emp_table
""")

employees_categorized.show()

+---+----+------+---------------+
| id|name|salary|salary_category|
+---+----+------+---------------+
|  1| Ali|  3000|            Low|
|  2|Sara|  7000|         Medium|
|  3|Omar|  null|        Unknown|
|  4|Mona| 10000|           High|
+---+----+------+---------------+

